# Cyberbullying model using LSTM

In [7]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import numpy as np
import string
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, Dropout, LSTM, Activation, Embedding
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.initializers import glorot_uniform


## Preprocessing the dataset

In [10]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
stop_words.update(list(string.punctuation))

In [11]:
df = pd.read_csv("bully.csv")
df.head() # displays only top 5 rows

,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying


In [12]:
messages = df['tweet_text'].astype(str)
# Convert labels to binary: 0 = not_cyberbullying, 1 = cyberbullying
y = df['cyberbullying_type'].apply(lambda x: 0 if str(x).strip().lower() == 'not_cyberbullying' else 1).values

In [13]:
df['tweet_text']

0        In other words #katandandre, your food was cra...
1        Why is #aussietv so white? #MKR #theblock #ImA...
2        @XochitlSuckkks a classy whore? Or more red ve...
3        @Jason_Gio meh. :P  thanks for the heads up, b...
4        @RudhoeEnglish This is an ISIS account pretend...
                               ...                        
47687    Black ppl aren't expected to do anything, depe...
47688    Turner did not withhold his disappointment. Tu...
47689    I swear to God. This dumb nigger bitch. I have...
47690    Yea fuck you RT @therealexel: IF YOURE A NIGGE...
47691    Bro. U gotta chill RT @CHILLShrammy: Dog FUCK ...
Name: tweet_text, Length: 47692, dtype: object

In [14]:
def get_simple_pos(tag) :
    if tag.startswith('J') : #ADJ
        return wordnet.ADJ
    elif tag.startswith('V') :#VERB
        return wordnet.VERB
    elif tag.startswith('N') :#NOUN
        return wordnet.NOUN
    elif tag.startswith('R') :#ADV
        return wordnet.ADV
    else:
        return wordnet.NOUN # default NOUN

def clean_text(review) :
    global max_len
    words = word_tokenize(review)
    output_words = []
    for word in words :
        if word.lower() not in stop_words :
            pos = pos_tag([word])
            clean_word = lemmatizer.lemmatize(word,pos = get_simple_pos(pos[0][1]))
            output_words.append(clean_word.lower())
    max_len = max(max_len, len(output_words))
    return " ".join(output_words)

In [15]:
max_len = 0 

In [16]:
print("Before:", messages[0])
messages = [clean_text(message) for message in messages]
print("After:", messages[0])


Before: In other words #katandandre, your food was crapilicious! #mkr
After: word katandandre food crapilicious mkr


In [17]:
def read_glove_vecs(glove_file):
    with open(glove_file, 'r', encoding="utf8") as file:
        word_to_vec_map = {}
        word_to_index = {}
        index_to_word = {}
        index = 0
        for line in file:
            line = line.strip().split()
            curr_word = line[0]
            word_to_index[curr_word] = index
            index_to_word[index] = curr_word
            word_to_vec_map[curr_word] = np.array(line[1:], dtype=np.float64)
            index += 1
    return word_to_index, index_to_word, word_to_vec_map

In [18]:

word_to_index, index_to_word, word_to_vec_map=read_glove_vecs('glove.6B.50d.txt')

In [19]:
def sentences_to_indices(X, word_to_index, max_len):
    m = len(X)
    X_indices = np.zeros((m, max_len))
    for i in range(m):
        sentence_words = [w.lower() for w in X[i].split()]
        for j, word in enumerate(sentence_words):
            if j >= max_len:  # stop if sentence is longer than max_len
                break
            if word in word_to_index:
                X_indices[i, j] = word_to_index[word]
    return X_indices


## The LSTM model

In [ ]:
def pretrained_embedding_layer(word_to_vec_map, word_to_index):
    vocab_len = len(word_to_index) + 1
    emb_dim = word_to_vec_map["cucumber"].shape[0]

    emb_matrix = np.zeros((vocab_len, emb_dim))

    for word, index in word_to_index.items():
        emb_matrix[index, :] = word_to_vec_map[word]

    embedding_layer = Embedding(vocab_len, emb_dim, trainable=True)
    embedding_layer.build((None,))
    embedding_layer.set_weights([emb_matrix])
    return embedding_layer

In [ ]:
# First, create the embedding layer once
embedding_layer = pretrained_embedding_layer(word_to_vec_map, word_to_index)

# Then pass it into the model
def NLPModel(input_shape, embedding_layer):
    sentence_indices = Input(input_shape, dtype='int32')
    embeddings = embedding_layer(sentence_indices)
    
    X = LSTM(64, return_sequences=False)(embeddings)
    X = Dense(1, activation='sigmoid')(X)    
    model = Model(inputs=sentence_indices, outputs=X)
    return model

In [22]:
max_len = 100

embedding_layer = pretrained_embedding_layer(word_to_vec_map, word_to_index)

model = NLPModel((max_len,), embedding_layer)

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [122]:
from sklearn.model_selection import train_test_split

In [123]:
X_train, X_test, Y_train, Y_test = train_test_split(messages, y, random_state = 0, test_size = 0.1)


In [124]:
X_train_indices = sentences_to_indices(X_train, word_to_index, max_len)

In [128]:
model.fit(X_train_indices, Y_train, epochs = 100, batch_size = 32, shuffle = True)

Epoch 1/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 248s 184ms/step - accuracy: 0.8327 - loss: 0.4525
Epoch 2/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 250s 187ms/step - accuracy: 0.8331 - loss: 0.4516
Epoch 3/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 251s 187ms/step - accuracy: 0.8331 - loss: 0.4516
Epoch 4/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 253s 188ms/step - accuracy: 0.8331 - loss: 0.4514
Epoch 5/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 252s 188ms/step - accuracy: 0.8331 - loss: 0.4512
Epoch 6/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 257s 191ms/step - accuracy: 0.8331 - loss: 0.4514
Epoch 7/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 254s 189ms/step - accuracy: 0.8331 - loss: 0.4512
Epoch 8/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 246s 183ms/step - accuracy: 0.8331 - loss: 0.4512
Epoch 9/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 245s 183ms/step - accuracy: 0.8331 - loss: 0.4512
Epoch 10/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 246s 183ms/step - accuracy: 0.8331 - loss: 0.4513
Epoch 11/100
1342/1342 ━━━━━━━━━━━━━━━━━━━━ 255s 190ms/step - accurac

In [ ]:
max_len = 100

embedding_layer = pretrained_embedding_layer(word_to_vec_map, word_to_index)

model = NLPModel((max_len,), embedding_layer)

model.summary()


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_16 (Embedding)        │ (None, 100, 50)        │    20,000,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_18 (LSTM)                  │ (None, 64)             │        29,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,029,555 (76.41 MB)

 Trainable params: 20,029,555 (76.41 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile (this does NOT retrain)
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Evaluate / test the already-trained model
test_loss, test_accuracy = model.evaluate(
    X_test_indices,
    Y_test,
    batch_size=32
)

print("Test Accuracy:", test_accuracy)


150/150 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8367 - loss: 0.5012
Test Accuracy: 0.8366876244544983


## Accuracy of LSTM: 83.67%

In [141]:
y_pred_prob = model.predict(X_test_indices, batch_size=32)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()

150/150 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step


In [142]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(Y_test, y_pred)
recall = recall_score(Y_test, y_pred)
f1 = f1_score(Y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Precision: 0.8366533864541833
Recall: 1.0
F1-score: 0.911062906724512


In [143]:
from sklearn.metrics import classification_report
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.00      0.00       780
           1       0.84      1.00      0.91      3990

    accuracy                           0.84      4770
   macro avg       0.92      0.50      0.46      4770
weighted avg       0.86      0.84      0.76      4770



In [144]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(Y_test, y_pred))

[[   1  779]
 [   0 3990]]


## Predictions

In [ ]:
text = "suck it"
text = [clean_text(text)]
text

In [135]:
text = sentences_to_indices(text, word_to_index, max_len)

In [ ]:
text

array([[23695.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.]])

In [138]:
model.predict(text)[0][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step


np.float32(0.69086695)

## Extras

In [139]:
import pickle

In [140]:
pickle.dump(word_to_index, open('word_to_index.pkl', 'wb'))